In [1]:
from utils_models import *

In [2]:
system = FluxoniumOscillatorSystem()

In [8]:

def dressed_to_2_level_dm(dressed_dm: qutip.Qobj, 
                        product_to_dressed: dict, 
                        qbt_position: int,
                        filtered_product_to_dressed: dict,
                        sign_multiplier:dict,
                        products_to_keep=None,
                        ) -> qutip.Qobj:
    """
    Convert a dressed density matrix to a multi-level density matrix for specified computational states,
    inferring subsystem dimensions from product_to_dressed.

    Parameters:
    - dressed_dm: The dressed density matrix as a qutip.Qobj.
    - product_to_dressed: Mapping from product states to dressed states indices.
    - qbt_position: which of the subsystem is the qubit
    - computational_0, computational_1: indicate which two levels of the qubit are the computational states.
    - products_to_keep: Optional list of product states to keep.

    Returns:
    - qutip.Qobj representing the reduced density matrix for specified computational states.
    """
    dressed_dm_data = pad_back_custom(dressed_dm, products_to_keep, product_to_dressed)
    if dressed_dm_data.shape[1] == 1:
        dressed_dm_data = qutip.ket2dm(dressed_dm_data)
    dressed_dm_data = dressed_dm_data.full()

    # Infer subsystem dimensions
    subsystem_dims = [max(indexes) + 1 for indexes in zip(*product_to_dressed.keys())]
    subsystem_dims[qbt_position] = 2
    rho_product = np.zeros((subsystem_dims*2), dtype=complex)
    for product_state, dressed_index1 in filtered_product_to_dressed.items():
        for product_state2, dressed_index2 in filtered_product_to_dressed.items():
            element = dressed_dm_data[dressed_index1, dressed_index2] * sign_multiplier[dressed_index1] * sign_multiplier[dressed_index2]
            rho_product[product_state+product_state2] += element
    print(rho_product)
    two_lvl_qbt_dm_size = np.prod(subsystem_dims)
    rho_product = rho_product.reshape((two_lvl_qbt_dm_size,two_lvl_qbt_dm_size))
    rho_product = qutip.Qobj(rho_product, dims=[subsystem_dims, subsystem_dims])

    rho_2_level = rho_product.ptrace(qbt_position)
    return rho_2_level



In [9]:
dressed_to_2_level_dm(qutip.ket2dm(qutip.basis(system.hilbertspace.dimension,2)),
                    system.product_to_dressed,
                    system.qbt_position,
                    system.filtered_product_to_dressed,
                    system.sign_multiplier,
                    system.products_to_keep)

Quantum object: dims = [[2], [2]], shape = (2, 2), type = oper, isherm = True
Qobj data =
[[0. 0.]
 [0. 1.]]